# Covariates and Shrinkage on the M5 Aggregate Series

This is stage 2 of the staged M5 study in #143: the checkpoint between the calendar covariates of stage 1 (#146) and the
stages that follow it, price and SNAP (stage 3) and stronger regularization (stage 4). Those two are filed only after
this notebook is approved on #148.

It fits the #118 default M5 model three ways on the ten series of `load_m5_aggregate()`. The default has a level state
and six weekday states that drift, a Yeo-Johnson target and half-normal priors on its sigmas. The three fits differ only
in the coefficients added to it:

| Fit | Added columns | Prior of each added coefficient |
|---|---|---|
| **no covariates** | none | none |
| **loose** | 74: annual Fourier, events, month, day of month, price index | $N(0, 1)$, the stage 1 prior |
| **tight** | the same 74 | $N(0, 0.1^2)$ |

Three things to keep in mind while reading:

- **This is not a scored stage.** The M5 score (WRMSSE over 30,490 item-store series) decides a stage in #143. This
  notebook only shows how covariates and shrinkage behave, on ten dense series and one 28-day window.
- **The covariates enter with fixed coefficients.** Their drift is 0, as in stage 1. Time-varying annual coefficients
  are out of scope for #143.
- **It is committed without outputs.** It reads only the packaged panel, so no raw M5 download is needed, and it runs in
  about a minute on a laptop CPU. It needs the `m5` extra: `pip install -e ".[m5]"`. Section 9 records the numbers of
  one run.

In [ ]:
import jax.numpy as jnp
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from bunobee.datasets import load_m5_aggregate
from bunobee.models.m5.m5_ssp_optim import SigmaPrior, fit_one_series_opt, predict_one_series_opt
from bunobee.models.m5.transforms import PowerTransform
from bunobee.models.ssp.kalman_1d import kalman_filter_1d
from bunobee.regression import make_periodic_dummies

plt.rcParams.update(
    {
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#e6e5e0",
        "grid.linewidth": 0.8,
        "axes.axisbelow": True,
        "font.size": 10,
        "axes.titlesize": 11,
        "lines.linewidth": 1.5,
    }
)

# One color per fit, the same in every figure. Actual sales are black.
FIT_COLORS = {"no covariates": "#2a78d6", "loose": "#eb6834", "tight": "#1baf7a"}
ACTUAL_COLOR = "#0b0b0b"
CONTEXT_COLOR = "#b5b4ad"
# The next two slots of the palette, for the two covariate groups of section 6.
GROUP_COLORS = {"annual + month": "#eda100", "price": "#e87ba4"}

## 1. The panel, and which days this notebook reads

`load_m5_aggregate()` holds 756 days of nine `state x category` sums and a national `TOTAL`. Its last 28 days, d_1914 to
d_1941, are M5's validation window, which is Kaggle's public leaderboard, and #143 reads that window only once per
candidate. So the notebook drops those days as soon as it loads the panel, and holds out the pre-validation window of
#125 instead, d_1886 to d_1913. That leaves 700 days to fit on and 28 to score.

In [ ]:
N_FIT = 700
HORIZON = 28
M5_FIRST_DATE = pd.Timestamp("2011-01-29")  # d_1

panel = load_m5_aggregate()
last_day_read = np.sort(panel["date"].unique())[N_FIT + HORIZON - 1]
panel = panel[panel["date"] <= last_day_read]  # No later cell can read the validation window.

SERIES = sorted(panel["series_id"].unique())
TOTAL = SERIES.index("TOTAL")
sales = panel.pivot(index="date", columns="series_id", values="sales")[SERIES].to_numpy(dtype=float)  # (days, series)
price = panel.pivot(index="date", columns="series_id", values="price_index")[SERIES].to_numpy(dtype=float)
days = panel[panel["series_id"] == SERIES[0]].set_index("date")  # The columns that hold one value per date.
dates = days.index
n_days = len(dates)

d_number = (dates - M5_FIRST_DATE).days + 1
assert (d_number[N_FIT], d_number[-1]) == (1886, 1913)
print(f"fit:     {dates[0].date()} to {dates[N_FIT - 1].date()}, d_{d_number[0]} to d_{d_number[N_FIT - 1]}")
print(f"holdout: {dates[N_FIT].date()} to {dates[-1].date()}, d_{d_number[N_FIT]} to d_{d_number[-1]}")
print(f"series:  {SERIES}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 5.5), sharex=True)
axes[0].plot(dates, sales[:, TOTAL], color=ACTUAL_COLOR, lw=0.8)
axes[0].set_title("TOTAL daily unit sales")
axes[0].set_ylabel("units")
axes[1].plot(dates, price, color=CONTEXT_COLOR, lw=0.8)
axes[1].plot(dates, price[:, TOTAL], color=ACTUAL_COLOR, lw=1.2)
axes[1].set_title("Price index of each series, TOTAL in black. It stays within about 2% of 1.")
axes[1].set_ylabel("index")
for ax in axes:
    ax.axvspan(dates[N_FIT], dates[-1], color="#f0efe9", zorder=0)
axes[0].annotate("holdout", (dates[N_FIT], 1), xycoords=("data", "axes fraction"), va="top", color="#52514e")
fig.tight_layout()
plt.show()

## 2. The design

The base columns are the default's: a column of ones for the level, then Tuesday to Sunday with Monday dropped. The
covariates are the stage 1 columns (`CalendarCovariates` in `playground/m5_prototype/m5_models.py`) built from the
panel's own columns, plus the price index, which the panel carries for stage 3:

- **Annual:** the sine and cosine of period 365.25 days.
- **Events:** one 0/1 column per name in `event_name_1` and `event_name_2`.
- **Month and day of month:** 0/1 columns with January and day 1 dropped.
- **Price:** the log of the series' `price_index`, standardized over the fit days.

A coefficient's prior only means something next to its column's scale, which is why the price column is standardized.
The log price index has a standard deviation of 0.15% to 1% by series, so with the raw column even an $N(0, 1)$
coefficient could barely move the target. Standardized, the coefficient is the effect of a one-standard-deviation price
move, in units of the target's standard deviation.

Price is the one column that differs by series. The design is therefore built per series, and the notebook fits with
`fit_one_series_opt` rather than `fit_batch_series_opt`, which shares one `Z`. Ten series make that cheap.

In [ ]:
base = make_periodic_dummies(n_days, period=7, phase=int(dates[0].dayofweek), intercept=True)
n_base = base.shape[1]

event_names = np.setdiff1d(days[["event_name_1", "event_name_2"]], [""]).tolist()
events = pd.DataFrame({n: (days["event_name_1"] == n) | (days["event_name_2"] == n) for n in event_names}).astype(float)
calendar = pd.concat(
    {
        "annual": days[["annual_sin_1", "annual_cos_1"]],
        "event": events,
        "month": pd.get_dummies(days["month_of_year"], prefix="month", dtype=float).iloc[:, 1:],
        "day of month": pd.get_dummies(days["day_of_month"], prefix="day", dtype=float).iloc[:, 1:],
    },
    axis=1,
)

log_price = np.log(price)
price_column = (log_price - log_price[:N_FIT].mean(axis=0)) / log_price[:N_FIT].std(axis=0)  # (days, series)

COV_GROUPS = np.array([*calendar.columns.get_level_values(0), "price"])
COV_NAMES = np.array([*calendar.columns.get_level_values(1), "log price index"])
N_COVARIATES = len(COV_NAMES)


def covariates_of(j):
    # The (days, 74) covariate columns of series j, unscaled.
    return np.column_stack([calendar.to_numpy(), price_column[:, j]])


pd.Series(COV_GROUPS).value_counts(sort=False).rename("columns").to_frame()

The holdout decides less than it may seem to. It has **no event day**, so the event columns reach its forecasts only
through what they take away from the other states, and it can't say whether their coefficients forecast well. Two events
are also seen once in the fit days. The Christmas column is close to empty too: `load_m5_aggregate()` repairs the
closure days, so the panel has no Christmas drop to learn.

In [ ]:
event_days = pd.DataFrame(
    {"fit days": events.iloc[:N_FIT].sum().astype(int), "holdout days": events.iloc[N_FIT:].sum().astype(int)}
)
print(f"event days in the holdout: {event_days['holdout days'].sum()}")
print(f"events seen once in the fit days: {event_days.index[event_days['fit days'] == 1].tolist()}")

## 3. The prior, and how to set it

The model of a series, on its standardized Yeo-Johnson scale, is

$$y_t = \ell_t + w_t^\top d_t + \beta^\top x_t + \varepsilon_t, \qquad \beta_j \sim N(0, \tau^2),$$

with the level $\ell_t$, the weekday states $w_t$ on the weekday columns $d_t$, and the fixed coefficients $\beta$ on
the covariates $x_t$.

$\tau$ is the only thing that changes between the loose and the tight fit. `fit_one_series_opt` has no argument for it,
since it gives every state `a0 = 0` and `P0 = 1`, but it needs none: a column $\tau x$ with a coefficient
$b \sim N(0, 1)$ is the same model as a column $x$ with a coefficient $\beta = \tau b \sim N(0, \tau^2)$. So the
notebook scales the covariate columns by $\tau$ and multiplies the fitted states back by $\tau$. The next cell checks
that the two forms agree when the filter is run both ways.

**The settings:**

- **Loose, $\tau = 1$:** the unscaled columns, so this is the stage 1 prior itself. The target has a standard deviation
  of 1 over the fit days, so a single coefficient may be as large as the whole series' variation.
- **Tight, $\tau = 0.1$:** one tenth of that. About 95% of the prior's mass lies within $\pm 0.2$ of the target's
  standard deviation, well inside the daily noise $\sigma_h$ of about 0.45. A coefficient must then have many days
  behind it to move far from zero.

Neither was tuned on the holdout. Section 8 sweeps $\tau$ and shows that 0.1 is tighter than the value both the fit days
and the holdout prefer. It stays as the tight setting so that the comparison shows the prior doing clearly too much.

In [ ]:
TRANSFORM = PowerTransform("yeo-johnson")
# The half-normal priors of the #118 default (HALF_NORMAL_PRIORS in playground/m5_prototype/m5_models.py).
SIGMA_PRIORS = {"sigma_h_prior": SigmaPrior("half_normal", (1.0,)), "sigma_q_prior": SigmaPrior("half_normal", (0.05,))}

FITS = {"no covariates": None, "loose": 1.0, "tight": 0.1}


def design(j, tau=None):
    # The design of series j over every day, with its covariates scaled by tau. tau=None leaves them out.
    return base if tau is None else np.column_stack([base, tau * covariates_of(j)])


# The check: P0 = tau^2 on the unscaled columns against P0 = 1 on the scaled ones, at fixed sigmas.
tau = FITS["tight"]
transform_params = TRANSFORM.fit(sales[:N_FIT, TOTAL])
y = jnp.asarray(TRANSFORM.forward(sales[:N_FIT, TOTAL], transform_params))
sigma_q = jnp.asarray(np.r_[0.05, np.full(n_base - 1, 0.005), np.zeros(N_COVARIATES)], dtype=jnp.float32)
n_states = n_base + N_COVARIATES
Z_plain = np.column_stack([base, covariates_of(TOTAL)]).astype(np.float32)
Z_scaled = design(TOTAL, tau).astype(np.float32)
run = dict(a0=jnp.zeros(n_states), sigma_h=0.45, sigma_q=sigma_q, y=y, logp=False)
P0_plain = jnp.concatenate([jnp.ones(n_base), jnp.full(N_COVARIATES, tau**2)])
_, a_plain, *_ = kalman_filter_1d(P0=P0_plain, Z=jnp.asarray(Z_plain[:N_FIT]), **run)
_, a_scaled, *_ = kalman_filter_1d(P0=jnp.ones(n_states), Z=jnp.asarray(Z_scaled[:N_FIT]), **run)

coef_gap = np.abs(np.asarray(a_plain[-1, n_base:]) - tau * np.asarray(a_scaled[-1, n_base:])).max()
forecast_gap = np.abs(Z_plain[N_FIT:] @ np.asarray(a_plain[-1]) - Z_scaled[N_FIT:] @ np.asarray(a_scaled[-1])).max()
print(f"largest gap of a coefficient: {coef_gap:.1e}, of a forecast mean: {forecast_gap:.1e}")
assert max(coef_gap, forecast_gap) < 1e-4

## 4. The three fits

Each series is fitted by MAP on its 700 fit days. Adam chooses the sigmas against the filter's likelihood, the filter
updates the coefficients day by day, and the forecast uses the coefficients filtered up to the last fit day. `coef` and
`coef_sd` below are those coefficients on the unscaled columns and their filtered standard deviations.

`loss` is the fit's negative log posterior of the sigmas. It only sees the fit days. It charges for a wide prior: until
the filter has learned a coefficient, a wide prior widens the predicted variance of the days its column touches, which
lowers the likelihood.

In [ ]:
def fit_series(j, tau=None):
    # One MAP fit of series j.
    Z = design(j, tau).astype(np.float32)
    fit = fit_one_series_opt(
        sales[:N_FIT, j],
        Z=jnp.asarray(Z[:N_FIT]),
        transform=TRANSFORM,
        fixed_states=None if tau is None else np.arange(Z.shape[1]) >= n_base,
        **SIGMA_PRIORS,
    )
    states = fit["at"][-1, n_base:]
    state_var = fit["Pt"][-1, n_base:]
    return {
        "forecast": predict_one_series_opt(fit, Z_future=Z[N_FIT:]),
        "level": fit["at"][:, 0],
        "sigma_h": fit["sigma_h"],
        "sigma_q_level": fit["sigma_q"][0],
        "sigma_q_weekday": fit["sigma_q"][1],
        "loss": fit["final_loss"],
        # Back on the unscaled columns: beta = tau * b.
        "coef": (tau or 1.0) * states,
        "coef_sd": (tau or 1.0) * np.sqrt(state_var),
    }


def fit_panel(tau=None):
    # Every series under one tau, each output stacked with the series first.
    by_series = [fit_series(j, tau) for j in range(len(SERIES))]
    return {key: np.stack([fit[key] for fit in by_series]) for key in by_series[0]}


results = {name: fit_panel(tau) for name, tau in FITS.items()}

## 5. Forecast accuracy on the holdout

MAPE is well defined on these dense series. The seasonal-naive column repeats the last fit week, the baseline of
`tests/test_m5_accuracy.py`. Bias is the error of the 28-day sum.

In [ ]:
actual = sales[N_FIT:]  # (horizon, series)
naive = sales[N_FIT - 7 : N_FIT][np.arange(HORIZON) % 7]


def mape(forecast):
    return 100 * np.mean(np.abs(forecast - actual) / actual, axis=0)


def bias(forecast):
    return 100 * (forecast.sum(axis=0) / actual.sum(axis=0) - 1)


forecasts = {name: result["forecast"].T for name, result in results.items()}
mape_table = pd.DataFrame({name: mape(f) for name, f in forecasts.items()} | {"seasonal naive": mape(naive)}, SERIES)
bias_table = pd.DataFrame({name: bias(f) for name, f in forecasts.items()}, SERIES)

accuracy = pd.concat({"MAPE %": mape_table, "bias %": bias_table}, axis=1)
accuracy.loc["panel mean"] = accuracy.mean()
accuracy.round(2)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 5.5), sharex=True)
context = 14  # Fit days drawn before the holdout.
for j, (ax, name) in enumerate(zip(axes.flat, SERIES)):
    ax.plot(dates[N_FIT - context :], sales[N_FIT - context :, j], color=ACTUAL_COLOR, lw=1.2, label="actual")
    for fit_name, forecast in forecasts.items():
        ax.plot(dates[N_FIT:], forecast[:, j], color=FIT_COLORS[fit_name], label=fit_name)
    ax.axvline(dates[N_FIT], color=CONTEXT_COLOR, lw=0.8)
    ax.set_title(name)
    ax.xaxis.set_major_locator(mdates.DayLocator(bymonthday=(1, 15)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %-d"))
axes.flat[0].set_ylabel("units")
axes.flat[5].set_ylabel("units")
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()

## 6. What the covariates change inside the model

**They take variation from the level and the noise.** The covariates explain moves the level would otherwise have to
follow, so its drift falls, and the noise falls with it. The weekday drift rises a little under the loose prior. The
table gives the median over the ten series.

In [ ]:
sigma_table = pd.DataFrame(
    {
        name: {
            "sigma_h": np.median(result["sigma_h"]),
            "sigma_q level": np.median(result["sigma_q_level"]),
            "sigma_q weekday": np.median(result["sigma_q_weekday"]),
        }
        for name, result in results.items()
    }
)
sigma_table.round(4)

In [ ]:
# The first 28 filtered days are left out: they mostly show the filter settling from its a0 = 0.
start = 28
total_columns = covariates_of(TOTAL)
loose_total = results["loose"]["coef"][TOTAL]
slow = np.isin(COV_GROUPS, ["annual", "month"])

fig, axes = plt.subplots(3, 1, figsize=(12, 8.5), sharex=True)
for name, result in results.items():
    axes[0].plot(dates[start:N_FIT], result["level"][TOTAL, start:], color=FIT_COLORS[name], label=name)
axes[0].set_title("TOTAL: filtered level. Compare how much it moves, not its height.")
axes[0].set_ylabel("target scale")
axes[0].legend(frameon=False, ncol=3)
for name in ("loose", "tight"):
    axes[1].plot(dates, total_columns @ results[name]["coef"][TOTAL], color=FIT_COLORS[name], lw=0.9, label=name)
axes[1].set_title("TOTAL: what all the covariates add on each day, at the coefficients of the last fit day")
axes[1].legend(frameon=False, ncol=2)
slow_part = total_columns[:, slow] @ loose_total[slow]
axes[2].plot(dates, slow_part, color=GROUP_COLORS["annual + month"], label="annual + month")
axes[2].plot(dates, total_columns[:, -1] * loose_total[-1], color=GROUP_COLORS["price"], label="price")
axes[2].set_title("TOTAL, loose prior: two of the groups behind that line")
axes[2].legend(frameon=False, ncol=2)
for ax in axes[1:]:
    ax.axvspan(dates[N_FIT], dates[-1], color="#f0efe9", zorder=0)
    ax.set_ylabel("target scale")
fig.tight_layout()
plt.show()

The height of the level differs between the fits because its meaning does: with covariates it is the level of a Monday
in January on day 1 of a month, not of an average day. Under the loose prior the level of `TOTAL` hardly moves, and the
slow columns carry what it did before.

**A tight prior shrinks every coefficient, but not by the same factor.** Below, each point is one coefficient of one
series, and a point on the dashed line was not shrunk at all. The annual, month and price coefficients fall by about ten
times, the ratio of the two values of $\tau$, so the prior sets their size more than the data does: they move as slowly
as the level, which can do the same job. Event coefficients fall about as far, for another reason: they have one or two
days each. Day-of-month coefficients lose far less. Each has about 23 days behind it, and they change too fast for the
level to stand in.

In [ ]:
groups = ["annual", "event", "month", "day of month", "price"]
loose_coef, tight_coef = results["loose"]["coef"], results["tight"]["coef"]

fig, axes = plt.subplots(1, len(groups), figsize=(15, 3.4))
for ax, group in zip(axes, groups):
    in_group = COV_GROUPS == group
    x, y = loose_coef[:, in_group].ravel(), tight_coef[:, in_group].ravel()
    limit = 1.05 * max(np.abs(x).max(), np.abs(y).max())
    ax.plot([-limit, limit], [-limit, limit], color=CONTEXT_COLOR, lw=0.8, ls="--")
    ax.scatter(x, y, s=16, color=FIT_COLORS["tight"], edgecolor="white", linewidth=0.5)
    ax.set(xlim=(-limit, limit), ylim=(-limit, limit), title=group, xlabel="loose coefficient")
    ax.set_aspect("equal")
axes[0].set_ylabel("tight coefficient")
fig.tight_layout()
plt.show()


def rms(values):
    return float(np.sqrt(np.mean(values**2)))


pd.DataFrame(
    {
        group: {
            "columns": int((COV_GROUPS == group).sum()),
            "RMS loose": rms(loose_coef[:, COV_GROUPS == group]),
            "RMS tight": rms(tight_coef[:, COV_GROUPS == group]),
            "tight / loose": rms(tight_coef[:, COV_GROUPS == group]) / rms(loose_coef[:, COV_GROUPS == group]),
        }
        for group in groups
    }
).T.round(3)

**Events show the cost of one prior for all.** These are the event coefficients of `TOTAL` with two filtered standard
deviations either side, and each name carries its number of fit days. With one or two days each, the loose intervals are
wide and most cross zero. Some effects are plain in the sales themselves: in both years `TOTAL` sold about a quarter
less on Thanksgiving than on the same weekday a week before, and about a quarter more on Labor Day. The tight prior
pulls these almost to zero along with the noise. A single $\tau$ cannot tell them apart, and this holdout can't charge
for it, since it has no event day.

In [ ]:
order = np.argsort(loose_coef[TOTAL, COV_GROUPS == "event"])
names = COV_NAMES[COV_GROUPS == "event"][order]
rows = np.arange(len(names))

fig, ax = plt.subplots(figsize=(8, 8.5))
for name, offset in (("loose", 0.17), ("tight", -0.17)):
    coef = results[name]["coef"][TOTAL, COV_GROUPS == "event"][order]
    sd = results[name]["coef_sd"][TOTAL, COV_GROUPS == "event"][order]
    ax.errorbar(coef, rows + offset, xerr=2 * sd, fmt="o", ms=4, lw=1, color=FIT_COLORS[name], label=name)
ax.axvline(0, color=ACTUAL_COLOR, lw=0.8)
ax.set_yticks(rows, [f"{name} ({event_days.loc[name, 'fit days']})" for name in names])
ax.set_xlabel("coefficient on the target scale")
ax.set_title("TOTAL: event coefficients, with fit days in brackets")
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
plt.show()

**Price is a preview of stage 3, and a warning.** Each coefficient is the effect of a one-standard-deviation price move
on the target scale, with the prior zero-centered and no sign constraint, as #143 asks. A demand curve predicts a
negative value. Under the loose prior most are positive, and the table shows why they can't be read as price effects: on
most series the index rises steadily over the fit days, so it mostly measures time, and the coefficient takes up trend
that the level would otherwise follow. Stage 3 divides each item's price by its own mean, which leaves this trend in
place.

The column also leaks slightly here: the index's basket and mean prices are fixed over the panel's first 728 days, which
include this holdout. The prices themselves are known ahead of the sales.

In [ ]:
price_position = np.flatnonzero(COV_GROUPS == "price")[0]
pd.DataFrame(
    {
        f"{name} {stat}": results[name][key][:, price_position]
        for name in ("loose", "tight")
        for stat, key in (("coef", "coef"), ("sd", "coef_sd"))
    }
    | {"corr with time": [np.corrcoef(price_column[:N_FIT, j], np.arange(N_FIT))[0, 1] for j in range(len(SERIES))]},
    SERIES,
).round(3)

## 7. A limit of these fits: the filter's covariance is diagonal

In the price table the `TOTAL` coefficient moves from 0.43 to 0.005 between the priors, which is more than 20 of its own
standard deviations. An estimate that sure of itself should not move that far. The reason is in `kalman_filter_1d`,
whose covariance is diagonal. It keeps one variance per state and drops the correlations between them, so it can't
record that a slow covariate and the level can stand in for each other. The fitter inherits that. Here the same filter
runs with the full covariance, in NumPy, at the sigmas each fit chose. It is not a refit: only the filter changes, and
the sigmas stay those of the diagonal filter.

In [ ]:
def exact_filter(j, tau, result):
    # The filter of kalman_filter_1d with the full state covariance, at the sigmas of series j in `result`.
    Z = design(j, tau).astype(float)
    params = TRANSFORM.fit(sales[:N_FIT, j])
    y = TRANSFORM.forward(sales[:N_FIT, j], params).astype(float)
    drift = np.zeros(Z.shape[1])
    drift[0], drift[1:n_base] = result["sigma_q_level"][j] ** 2, result["sigma_q_weekday"][j] ** 2
    noise = float(result["sigma_h"][j]) ** 2
    a, P = np.zeros(Z.shape[1]), np.eye(Z.shape[1])
    for z, value in zip(Z[:N_FIT], y):
        Pz = P @ z
        F = z @ Pz + noise
        a = a + Pz * (value - z @ a) / F
        P = P - np.outer(Pz, Pz) / F + np.diag(drift)
    Z_future = Z[N_FIT:]
    forecast = TRANSFORM.inverse(Z_future @ a, np.sum((Z_future @ P) * Z_future, axis=1) + noise, params)
    scale = tau or 1.0
    return {"forecast": forecast, "coef": scale * a[n_base:], "coef_sd": scale * np.sqrt(np.diag(P)[n_base:])}


exact = {}
for name, tau in FITS.items():
    by_series = [exact_filter(j, tau, results[name]) for j in range(len(SERIES))]
    exact[name] = {key: np.stack([fit[key] for fit in by_series]) for key in by_series[0]}

pd.DataFrame(
    {
        "MAPE % diagonal": {name: mape(forecasts[name]).mean() for name in FITS},
        "MAPE % full": {name: mape(exact[name]["forecast"].T).mean() for name in FITS},
        "bias % diagonal": {name: bias(forecasts[name]).mean() for name in FITS},
        "bias % full": {name: bias(exact[name]["forecast"].T).mean() for name in FITS},
    }
).round(2)

In [ ]:
# Coefficients and their spread under the loose prior, where the two filters differ most. Medians over all ten series.
loose_diagonal, loose_full = results["loose"], exact["loose"]
rows = {}
for group in groups:
    in_group = COV_GROUPS == group
    rows[group] = {
        "sd diagonal": np.median(loose_diagonal["coef_sd"][:, in_group]),
        "sd full": np.median(loose_full["coef_sd"][:, in_group]),
        "RMS coef diagonal": rms(loose_diagonal["coef"][:, in_group]),
        "RMS coef full": rms(loose_full["coef"][:, in_group]),
    }
print(pd.DataFrame(rows).T.round(3))
print()
print(
    pd.DataFrame(
        {
            "diagonal": loose_diagonal["coef"][:, price_position],
            "diagonal sd": loose_diagonal["coef_sd"][:, price_position],
            "full": loose_full["coef"][:, price_position],
            "full sd": loose_full["coef_sd"][:, price_position],
        },
        SERIES,
    ).round(3)
)

## 8. The whole path from tight to loose

Two fits only show two points of a curve, so this section fits the panel on a grid of $\tau$, with the diagonal filter
of the fits above. Two scores are drawn, on separate axes. The holdout MAPE reads the 28 days held out. The summed
`loss` of section 4 reads only the fit days, so it could choose $\tau$ without touching any held-out day.

As $\tau$ goes to 0 the fit must return to the no-covariates fit, on both scores. That is a check on the column scaling
as well as an illustration.

In [ ]:
TAU_GRID = [1.0, 0.5, 0.3, 0.2, 0.1, 0.05, 0.02, 0.01]
sweep = {tau: fit_panel(tau) for tau in TAU_GRID}


def scores(result):
    forecast = result["forecast"].T
    return {
        "panel MAPE %": mape(forecast).mean(),
        "mean bias %": bias(forecast).mean(),
        "summed loss": result["loss"].sum(),
        "median sigma_q level": np.median(result["sigma_q_level"]),
    }


sweep_table = pd.DataFrame({tau: scores(result) for tau, result in sweep.items()}).T.rename_axis("tau")
no_covariates = scores(results["no covariates"])
sweep_table.loc["no covariates"] = no_covariates
sweep_table.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharex=True)
for ax, column in zip(axes, ("panel MAPE %", "summed loss")):
    values = [scores(sweep[tau])[column] for tau in TAU_GRID]
    ax.plot(TAU_GRID, values, color=ACTUAL_COLOR, lw=1.2, marker="o", ms=4, zorder=2)
    ax.axhline(no_covariates[column], color=FIT_COLORS["no covariates"], ls="--", label="no covariates")
    for name in ("loose", "tight"):
        value = scores(sweep[FITS[name]])[column]
        ax.scatter(FITS[name], value, s=70, color=FIT_COLORS[name], edgecolor="white", zorder=3, label=name)
    ax.set_xscale("log")
    ax.set_xlabel(r"prior standard deviation $\tau$")
    ax.set_title(column + ", lower is better")
    ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## 9. What this run shows

Measured on 2026-11-07 on a macOS CPU, at float32. The notebook is committed without outputs, so this cell is the record
of them. A rerun can differ in the last digits. Panel means over the ten series, on the 28-day pre-validation holdout:

| Fit | MAPE % | Bias % | Summed loss | Series better than no covariates | Median level drift |
|---|---|---|---|---|---|
| no covariates | 7.76 | −3.52 | 5990.9 | | 0.105 |
| loose, $\tau = 1$ | 7.05 | +1.63 | 6211.0 | 6 of 10 | 0.037 |
| tight, $\tau = 0.1$ | 7.31 | −2.24 | 5739.9 | 5 of 10 | 0.060 |

Seasonal naive scores 10.77, so all three fits beat it.

- **Covariates help on these series, and neither prior is the best one.** Both fits beat no covariates on the panel
  mean, on six and five of the ten series. The sweep has its minimum between them: $\tau = 0.3$ scores 6.47 and has the
  lowest loss, 5525.7. The loss sees only the fit days, so the two scores agree independently. As $\tau$ falls to 0.01
  the fit returns to no covariates (7.75 against 7.76), as it must.
- **This does not carry over to M5 as it stands.** The calendar covariates under this loose prior made the full M5 run
  worse in stage 1 (WRMSSE 0.86564 against 0.79563, #146), most of all at the top of the hierarchy. This notebook has
  ten dense series and one window, and it doesn't explain the difference.
- **The window can't judge events.** It has no event day, so the forecast cost of shrinking events never shows in its
  MAPE. Thanksgiving on `TOTAL` goes from −2.28 under the loose prior to −0.25 under the tight one.
- **The price index mostly measures time.** It correlates at 0.74 to 0.97 with time on seven of the ten series. There
  the loose coefficients are positive, the wrong sign for a demand curve, and they take up trend from the level.
- **The filter's diagonal covariance matters for the coefficients more than for the forecasts.** Against a full
  covariance at the same sigmas under the loose prior, its standard deviations are too narrow by about 6 times for the
  annual columns, 3 times for month and 3 times for price (medians 0.032 against 0.187, 0.079 against 0.250, 0.024
  against 0.082). The RMS price coefficient falls from 0.287 to 0.088. The panel MAPE moves much less, and the order of
  the three fits stays: 7.79, 6.84 and 7.12.

**For the approval on #148,** before stages 3 and 4 are filed (`/plan-to-issues 143`):

1. Are these the loose and tight settings to keep, or should the comparison use another pair, such as 1 and 0.3?
2. Stage 3 encodes price over the series' own mean, which leaves the trend above in place. Should it remove the trend,
   or is the item-level price different enough to try it as decided?
3. Stage 4 will shrink through this filter. Should its choice of method account for the diagonal covariance first? One
   shared $\tau$ also can't separate Thanksgiving from noise, which favors a method that treats events one by one.